# Wikipedia earthquake tables (2000–2025)

This notebook retrieves the yearly Wikipedia pages from `List_of_earthquakes_in_2000` through `List_of_earthquakes_in_2025` and extracts their wiki tables with `pandas.read_html`.

**Output layout**

```text
Project/
├── raw/
│   └── wikipedia_earthquake_list/
│       ├── List_of_earthquakes_in_2000.html
│       └── ...
└── interim/
    └── wikipedia_earthquake_list/
        ├── earthquakes_2000/
        │   ├── By_death_toll_2000.csv
        │   ├── By_magnitude_2000.csv
        │   └── March_2000.csv
        ├── earthquakes_2001/
        └── ...
```

Each table file is named using its caption or the nearest section heading followed by the year. If multiple tables have the same heading, a `part2`, `part3`, etc. suffix is added. There are no collection dates in data filenames.

The original page HTML is stored unchanged in `raw/`. Parsed tables are saved to `interim/`. This notebook does not clean or impute table values; values such as `-`, death-toll ranges, and merged-cell repeats should be handled in the cleaning stage.

**Source:** https://en.wikipedia.org/wiki/List_of_earthquakes_in_2000 (the year in the URL changes from 2000 to 2025). Wikipedia content is generally available under CC BY-SA; retain attribution to the specific pages and check applicable terms. The notebook checks `robots.txt` before retrieval.

## 1. Imports and folders

In [9]:
from pathlib import Path
from io import StringIO
from urllib.robotparser import RobotFileParser
from datetime import datetime, timezone
from collections import defaultdict
import re
import time

import requests
import pandas as pd
from bs4 import BeautifulSoup

# Run this notebook from the EarthQuake project root.
PROJECT_DIR = Path.cwd()


RAW_DIR = PROJECT_DIR / "raw"
INTERIM_DIR = PROJECT_DIR / "interim"
REPORT_DIR = PROJECT_DIR / "reports"

RAW_WIKI_DIR = RAW_DIR / "wikipedia_earthquake_list"
WIKI_INTERIM_DIR = INTERIM_DIR / "wikipedia_earthquake_list"

for folder in [RAW_WIKI_DIR, WIKI_INTERIM_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

START_YEAR, END_YEAR = 2000, 2025
PAGE_URL = "https://en.wikipedia.org/wiki/List_of_earthquakes_in_{year}"
ROBOTS_URL = "https://en.wikipedia.org/robots.txt"
USER_AGENT = "EarthquakeAcademicProject/1.0 (student research)"
REQUEST_DELAY_SECONDS = 1.0
MAX_ATTEMPTS = 3

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})

print("Project:", PROJECT_DIR)
print("Raw HTML folder:", RAW_WIKI_DIR)
print("Interim tables folder:", WIKI_INTERIM_DIR)
print("Years:", START_YEAR, "to", END_YEAR)


Project: d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes
Raw HTML folder: d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\raw\wikipedia_earthquake_list
Interim tables folder: d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\interim\wikipedia_earthquake_list
Years: 2000 to 2025


## 2. Check Wikipedia robots.txt

In [10]:
robots_response = session.get(ROBOTS_URL, timeout=(15, 60))
robots_response.raise_for_status()

# Keep a stable filename: no collection date in the filename.
robots_path = REPORT_DIR / "robots_wikipedia.txt"
robots_path.write_text(
    f"Requested URL: {ROBOTS_URL}\n"
    f"HTTP status: {robots_response.status_code}\n"
    f"Checked at UTC: {datetime.now(timezone.utc).isoformat()}\n\n"
    + robots_response.text,
    encoding="utf-8",
)

robots_parser = RobotFileParser()
robots_parser.parse(robots_response.text.splitlines())

sample_url = PAGE_URL.format(year=2000)
robots_allowed = robots_parser.can_fetch(USER_AGENT, sample_url)

print("robots.txt saved to:", robots_path)
print("HTTP status:", robots_response.status_code)
print("Example page:", sample_url)
print("Allowed by robots rules:", robots_allowed)
print("Terms of Use: https://foundation.wikimedia.org/wiki/Policy:Terms_of_Use")
print("User-Agent policy: https://foundation.wikimedia.org/wiki/Policy:User-Agent_policy")

if not robots_allowed:
    raise RuntimeError("Wikipedia robots.txt does not allow the target page for this user agent.")

print("\n".join(robots_response.text.splitlines()[:15]))



robots.txt saved to: d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\reports\robots_wikipedia.txt
HTTP status: 200
Example page: https://en.wikipedia.org/wiki/List_of_earthquakes_in_2000
Allowed by robots rules: True
Terms of Use: https://foundation.wikimedia.org/wiki/Policy:Terms_of_Use
User-Agent policy: https://foundation.wikimedia.org/wiki/Policy:User-Agent_policy
﻿# robots.txt for http://www.wikipedia.org/ and friends
#
# Please note: There are a lot of pages on this site, and there are
# some misbehaved spiders out there that go _way_ too fast. If you're
# irresponsible, your access to the site may be blocked.
#

# Observed spamming large amounts of https://en.wikipedia.org/?curid=NNNNNN
# and ignoring 429 ratelimit responses, claims to respect robots:
# http://mj12bot.com/
User-agent: MJ12bot
Disallow: /

# advertising-related bots:
User-agent: Mediapartners-Google*


## 3. Retrieve the yearly pages and save raw HTML

In [11]:
request_log = []

for year in range(START_YEAR, END_YEAR + 1):
    url = PAGE_URL.format(year=year)
    raw_path = RAW_WIKI_DIR / f"List_of_earthquakes_in_{year}.html"

    if not robots_parser.can_fetch(USER_AGENT, url):
        request_log.append({
            "year": year, "url": url, "status": "disallowed",
            "raw_file": "", "retrieved_at_utc": "",
        })
        print(year, "disallowed by robots.txt")
        continue

    if raw_path.exists() and raw_path.stat().st_size > 0:
        request_log.append({
            "year": year, "url": url, "status": "reused cached HTML",
            "raw_file": str(raw_path), "retrieved_at_utc": "",
        })
        print(year, "reused cached HTML")
        continue

    success = False
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            response = session.get(url, timeout=(15, 60))
            if response.status_code == 404:
                request_log.append({
                    "year": year, "url": url, "status": "404",
                    "raw_file": "",
                    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
                })
                print(year, "404 - page not found")
                success = True
                break

            response.raise_for_status()
            raw_path.write_bytes(response.content)  # unchanged response bytes

            request_log.append({
                "year": year, "url": url, "status": "downloaded",
                "raw_file": str(raw_path),
                "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
            })
            print(year, "downloaded")
            success = True
            time.sleep(REQUEST_DELAY_SECONDS)
            break

        except requests.RequestException as exc:
            print(f"{year}: attempt {attempt}/{MAX_ATTEMPTS} failed: {exc}")
            if attempt == MAX_ATTEMPTS:
                request_log.append({
                    "year": year, "url": url, "status": f"failed: {exc}",
                    "raw_file": "",
                    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
                })
            else:
                time.sleep(2 * attempt)

    if not success:
        print(year, "could not be downloaded; continuing to next year")

# The log records retrieval timestamps for the assignment, but filenames have no dates.
request_log_path = REPORT_DIR / "wikipedia_earthquake_request_log.csv"
pd.DataFrame(request_log).to_csv(request_log_path, index=False)
print("Request log:", request_log_path)


2000 downloaded
2001 downloaded
2002 downloaded
2003 downloaded
2004 downloaded
2005 downloaded
2006 downloaded
2007 downloaded
2008 downloaded
2009 downloaded
2010 downloaded
2011 downloaded
2012 downloaded
2013 downloaded
2014 downloaded
2015 downloaded
2016 downloaded
2017 downloaded
2018 downloaded
2019 downloaded
2020 downloaded
2021 downloaded
2022 downloaded
2023 downloaded
2024 downloaded
2025 downloaded
Request log: d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\reports\wikipedia_earthquake_request_log.csv


## 4. Extract tables with `pandas.read_html` and save by year

Only Wikipedia `wikitable` tables are extracted, so navigation and infobox tables are not included. The table name comes from its caption when present, otherwise from the nearest heading above it. If no suitable name exists, it receives a stable `table_XX` name. Multi-row column headers are flattened for CSV readability; table values are not cleaned.

In [12]:
def safe_filename(text):
    """Make a short, Windows-safe filename component."""
    text = re.sub(r"\[.*?\]", "", str(text))
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_")
    return text[:100] or "table"


def flatten_columns(columns):
    """Flatten a pandas MultiIndex header without changing table values."""
    flattened = []
    for column in columns:
        parts = column if isinstance(column, tuple) else (column,)
        parts = [
            str(part).strip()
            for part in parts
            if str(part).strip() and not str(part).startswith("Unnamed")
        ]
        flattened.append(" ".join(dict.fromkeys(parts)) or "column")

    seen = defaultdict(int)
    unique = []
    for name in flattened:
        seen[name] += 1
        unique.append(name if seen[name] == 1 else f"{name}_{seen[name]}")
    return unique


def table_label(table_tag, table_number):
    """Use a caption or nearest section heading as the table name."""
    caption = table_tag.find("caption", recursive=False)
    if caption:
        label = caption.get_text(" ", strip=True)
        if label:
            return safe_filename(label)

    heading = table_tag.find_previous(["h2", "h3", "h4"])
    if heading:
        for edit in heading.select(".mw-editsection"):
            edit.decompose()
        label = heading.get_text(" ", strip=True)
        if label:
            return safe_filename(label)

    return f"table_{table_number:02d}"


extraction_report = []

for year in range(START_YEAR, END_YEAR + 1):
    url = PAGE_URL.format(year=year)
    raw_path = RAW_WIKI_DIR / f"List_of_earthquakes_in_{year}.html"
    year_dir = WIKI_INTERIM_DIR / f"earthquakes_{year}"

    if not raw_path.exists():
        extraction_report.append({
            "year": year, "url": url, "tables_found": 0,
            "tables_saved": 0, "status": "raw HTML missing",
            "output_folder": str(year_dir),
        })
        print(year, "skipped: raw HTML missing")
        continue

    html = raw_path.read_text(encoding="utf-8", errors="replace")
    soup = BeautifulSoup(html, "lxml")

    # Remove visible edit links and reference markers from table headers only.
    for tag in soup.select(".mw-editsection, sup.reference, span.sortkey"):
        tag.decompose()

    table_tags = [
        table for table in soup.select("table.wikitable")
        if table.find_parent("table") is None
    ]

    year_dir.mkdir(parents=True, exist_ok=True)
    name_counts = defaultdict(int)
    saved_count = 0

    for table_number, table_tag in enumerate(table_tags, start=1):
        try:
            parsed_tables = pd.read_html(
                StringIO(str(table_tag)),
                keep_default_na=False,
                thousands=None,
            )
        except (ValueError, ImportError) as exc:
            print(f"{year}: table {table_number} skipped: {exc}")
            continue

        if not parsed_tables:
            continue

        df = parsed_tables[0]
        df.columns = flatten_columns(df.columns)

        # Add the year as provenance metadata, unless already present.
        if not any(str(col).strip().lower() == "year" for col in df.columns):
            df.insert(0, "year", year)

        base_name = table_label(table_tag, table_number)
        name_counts[base_name] += 1
        suffix = "" if name_counts[base_name] == 1 else f"_part{name_counts[base_name]}"
        output_path = year_dir / f"{base_name}{suffix}_{year}.csv"

        try:
            df.to_csv(output_path, index=False, encoding="utf-8-sig")
        except PermissionError as exc:
            raise PermissionError(
                f"Cannot write {output_path}. Close that CSV in Excel and rerun this cell."
            ) from exc

        saved_count += 1
        extraction_report.append({
            "year": year,
            "url": url,
            "table_number_on_page": table_number,
            "table_name": base_name,
            "rows": len(df),
            "columns": len(df.columns),
            "column_names": " | ".join(map(str, df.columns)),
            "output_file": str(output_path),
            "status": "saved",
        })

    print(f"{year}: {len(table_tags)} wikitables found; {saved_count} saved to {year_dir}")

extraction_report_path = REPORT_DIR / "wikipedia_earthquake_extraction_report.csv"
pd.DataFrame(extraction_report).to_csv(extraction_report_path, index=False)
print("Extraction report:", extraction_report_path)


2000: 15 wikitables found; 15 saved to d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\interim\wikipedia_earthquake_list\earthquakes_2000
2001: 15 wikitables found; 15 saved to d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\interim\wikipedia_earthquake_list\earthquakes_2001
2002: 15 wikitables found; 15 saved to d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\interim\wikipedia_earthquake_list\earthquakes_2002
2003: 15 wikitables found; 15 saved to d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\interim\wikipedia_earthquake_list\earthquakes_2003
2004: 15 wikitables found; 15 saved to d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\interim\wikipedia_earthquake_list\earthquakes_2004
2005: 15 wikitables found; 15 saved to d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\interim\wikipedia_earthquake_list\earthquakes_2005
2006: 3 wikitables found; 3 saved to d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\interim\wikipedia_earthquake_list\earthquakes

## 5. Inspect the saved output (before cleaning)

In [13]:
# Show counts and inspect one saved table without loading all yearly CSVs.

report_df = pd.DataFrame(extraction_report)
if not report_df.empty:
    saved_summary = (
        report_df[report_df["status"] == "saved"]
        .groupby("year", as_index=False)
        .agg(tables_saved=("output_file", "count"))
    )
    display(saved_summary)

csv_files = sorted(WIKI_INTERIM_DIR.glob("earthquakes_*/*.csv"))
print("Total saved table CSV files:", len(csv_files))

if csv_files:
    sample_path = csv_files[0]
    sample_df = pd.read_csv(sample_path, keep_default_na=False)
    print("Sample file:", sample_path)
    print("Shape:", sample_df.shape)
    print("Columns:", sample_df.columns.tolist())
    print("Dtypes:\n", sample_df.dtypes)
    display(sample_df.head())

print("\nOriginal HTML files are in:", RAW_WIKI_DIR)
print("Extracted CSV tables are in:", WIKI_INTERIM_DIR)


,year,tables_saved
0,2000,15
1,2001,15
2,2002,15
3,2003,15
4,2004,15
5,2005,15
6,2006,3
7,2007,10
8,2008,3
9,2009,15


Total saved table CSV files: 296
Sample file: d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\interim\wikipedia_earthquake_list\earthquakes_2000\April_2000.csv
Shape: (9, 9)
Columns: ['year', 'Date', 'Country and location', 'Mw', 'Depth (km)', 'MMI', 'Notes', 'Casualties Dead', 'Casualties Injured']
Dtypes:
 year                    int64
Date                      str
Country and location      str
Mw                        str
Depth (km)                str
MMI                       str
Notes                     str
Casualties Dead           str
Casualties Injured        str
dtype: object


,year,Date,Country and location,Mw,Depth (km),MMI,Notes,Casualties Dead,Casualties Injured
0,2000,3,"Indonesia, North Sulawesi offshore, 147 km (91...",6.2,150.0,-,-,-,-
1,2000,6,"India, Maharashtra, 19 km (12 mi) southeast of...",4.9,10.0,VI,Two people injured and 150 homes damaged in th...,-,2
2,2000,7,"Mauritius, Rodrigues offshore, 286 km (178 mi)...",6.3,10.0,-,-,-,-
3,2000,18,"Tonga, Tongatapu offshore, 141 km (88 mi) west...",6.0,220.7,I,-,-,-
4,2000,21,"United States, Alaska offshore, 115 km (71 mi)...",6.0,33.0,I,-,-,-



Original HTML files are in: d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\raw\wikipedia_earthquake_list
Extracted CSV tables are in: d:\Pandu\IIT_Madras\DA2402\DA2402_Project_Earthquakes\interim\wikipedia_earthquake_list


## Notes for the cleaning stage

- These CSVs contain extracted Wikipedia table values, not cleaned analytical records.
- `-`, death-toll ranges, superscripts converted to text, missing values and repeated merged-cell values may need review.
- Different years can contain different numbers or layouts of tables. The extraction report records the tables saved from each page.
- Wikipedia's earthquake lists use UTC dates and state their own event-selection criteria. Cite the page URL when presenting extracted data and follow the applicable CC BY-SA attribution requirements.